In [ ]:
import sys
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import openpyxl
from PIL import Image

print("Python location:", sys.executable)
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("Notebook packages loaded successfully.")

In [ ]:
# 70 clear images, 70 blurry images, 60 low-light images, 50 text images = 250 images
from pathlib import Path
import pandas as pd

# Find the repository whether the notebook starts
# in blv-project or in its notebooks folder.
current_folder = Path.cwd().resolve()

PROJECT_FOLDER = next(
    (
        folder
        for folder in [current_folder, *current_folder.parents]
        if (folder / ".git").exists()
        and (folder / "outputs").is_dir()
    ),
    None
)

if PROJECT_FOLDER is None:
    raise FileNotFoundError(
        "Could not find the blv-project repository and outputs folder."
    )

OUTPUT_FOLDER = PROJECT_FOLDER / "outputs"
FINAL_FOLDER = OUTPUT_FOLDER / "final_sample"
PILOT_FOLDER = OUTPUT_FOLDER / "blur_calibration"

print("Project folder:", PROJECT_FOLDER)

# Read the completed sample without changing it.
sample_file = FINAL_FOLDER / "sample_manifest_with_blur.csv"

if not sample_file.is_file():
    raise FileNotFoundError(f"Missing file: {sample_file}")

sample = pd.read_csv(sample_file)

assert len(sample) == 250, "Expected 250 image records."
assert sample["file_name"].is_unique, "Duplicate filenames found."

print("Selected images:", len(sample))
print("Images with blur scores:", sample["laplacian_variance"].notna().sum())
print("\nGroup counts:")
print(sample["stratum"].value_counts())

In [ ]:
import json

# Read the pilot scores and the threshold already chosen
pilot = pd.read_csv(
    PILOT_FOLDER / "pilot_with_blur_scores.csv"
)

with open(PILOT_FOLDER / "selected_threshold.json") as file:
    threshold_settings = json.load(file)

# Check that the pilot records are complete
assert len(pilot) == 200, "Expected 200 pilot images."
assert pilot["file_name"].is_unique, "Duplicate pilot images found."
assert pilot["laplacian_variance"].notna().all(), "Missing blur scores."

# Check that the pilot and final sample contain different images
overlap = set(pilot["file_name"]) & set(sample["file_name"])
assert not overlap, f"Found {len(overlap)} overlapping images."

print("Pilot images:", len(pilot))
print("Pilot images with blur scores:", pilot["laplacian_variance"].notna().sum())
print("Overlap with final sample:", len(overlap))
print("Saved threshold:", threshold_settings["threshold"])

print("\nPilot groups:")
print(pd.crosstab(pilot["purpose"], pilot["reference_group"]))

In [ ]:
from pathlib import Path

for item in Path("/content/drive").iterdir():
    print(item)

In [ ]:
PROJECT_FOLDER = Path("/content/drive/MyDrive/NAIJA JOLLOF/BLV Project")
print(PROJECT_FOLDER.exists())

In [ ]:
for file in PROJECT_FOLDER.rglob("*.json"):
    print(file)

In [ ]:
import json
from pathlib import Path

ANNOTATION_FOLDER = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project/raw_data/annotations"
)

for file_name in ["train.json", "val.json", "test.json"]:
    file_path = ANNOTATION_FOLDER / file_name

    with open(file_path, "r") as file:
        data = json.load(file)

    print("\nFILE:", file_name)
    print("Data type:", type(data))

    if isinstance(data, dict):
        print("Top-level keys:", data.keys())

        if "images" in data:
            print("First image record:", data["images"][0])

        if "annotations" in data:
            print("First annotation record:", data["annotations"][0])

    elif isinstance(data, list):
        print("First record:", data[0])

In [ ]:
from pathlib import Path
import zipfile

RAW_DATA = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project/raw_data"
)

ZIP_FILE = RAW_DATA / "annotations.zip"
QUALITY_FOLDER = RAW_DATA / "quality_annotations"

QUALITY_FOLDER.mkdir(exist_ok=True)

with zipfile.ZipFile(ZIP_FILE, "r") as zip_file:
    print(zip_file.namelist())  # Shows the files before extracting
    zip_file.extractall(QUALITY_FOLDER)

print("Quality files extracted to:", QUALITY_FOLDER)

In [ ]:
import json
from pathlib import Path

QUALITY_TRAIN = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project/raw_data/quality_annotations/train.json"
)

with open(QUALITY_TRAIN, "r") as file:
    quality_data = json.load(file)

print("Data type:", type(quality_data))
print("First record:", quality_data[0])

In [ ]:
from pathlib import Path

RAW_DATA = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project/raw_data"
)

for file in sorted(RAW_DATA.rglob("*.json")):
    print(file)

In [ ]:
import json

for file in sorted(RAW_DATA.rglob("*.json")):
    with open(file, "r") as opened_file:
        data = json.load(opened_file)

    if isinstance(data, dict) and "images" in data:
        print("CAPTION FILE:", file)

    elif isinstance(data, list) and len(data) > 0 and "flaws" in data[0]:
        print("QUALITY FILE:", file)

In [ ]:
from pathlib import Path
import json
import pandas as pd

# Main project locations
RAW_DATA = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project/raw_data"
)

CAPTION_FOLDER = RAW_DATA / "raw_data caption_annotations"
QUALITY_FOLDER = RAW_DATA / "quality_annotations"

OUTPUT_FOLDER = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project/outputs"
)

OUTPUT_FOLDER.mkdir(exist_ok=True)

def load_caption_file(file_path, split_name):
    """Loads image details and valid human captions."""

    with open(file_path, "r") as file:
        data = json.load(file)

    # One row for each image
    images = pd.DataFrame(data["images"])[
        ["id", "file_name", "text_detected"]
    ].copy()

    images = images.rename(columns={"id": "image_id"})
    images["split"] = split_name

    # One row for each human-written caption
    captions = pd.DataFrame(data["annotations"]).copy()
    captions["split"] = split_name

    # Remove rejected and generic prewritten captions
    captions = captions[
        (~captions["is_rejected"]) &
        (~captions["is_precanned"])
    ][["image_id", "caption", "split"]]

    return images, captions


def load_quality_file(file_path, split_name):
    """Loads quality-vote information for each image."""

    with open(file_path, "r") as file:
        data = json.load(file)

    quality = pd.DataFrame(data).copy()

    # Turn the flaws dictionary into separate columns:
    # BLR, DRK, NON, BRT, FRM, OBS, OTH, and ROT
    flaw_votes = pd.json_normalize(quality["flaws"]).fillna(0)

    quality = pd.concat(
        [quality.drop(columns=["flaws"]), flaw_votes],
        axis=1
    )

    quality = quality.rename(columns={
        "image": "file_name",
        "unrecognizable": "unrecognizable_votes"
    })

    quality["split"] = split_name

    return quality


# Load caption information for train and validation only
train_images, train_captions = load_caption_file(
    CAPTION_FOLDER / "train.json",
    "train"
)

val_images, val_captions = load_caption_file(
    CAPTION_FOLDER / "val.json",
    "val"
)

# Load quality information for train and validation only
train_quality = load_quality_file(
    QUALITY_FOLDER / "train.json",
    "train"
)

val_quality = load_quality_file(
    QUALITY_FOLDER / "val.json",
    "val"
)

# Combine train and validation
images = pd.concat([train_images, val_images], ignore_index=True)
quality = pd.concat([train_quality, val_quality], ignore_index=True)
captions_long = pd.concat([train_captions, val_captions], ignore_index=True)

# Join caption-image information with quality information
master_metadata = images.merge(
    quality,
    on=["file_name", "split"],
    how="left",
    validate="one_to_one"
)

# Add filenames to each human caption for later CIDEr evaluation
captions_long = captions_long.merge(
    images[["image_id", "file_name", "split"]],
    on=["image_id", "split"],
    how="left",
    validate="many_to_one"
)

# Check whether any quality data failed to match
missing_quality = master_metadata["BLR"].isna().sum()

print("Number of train + validation images:", len(master_metadata))
print("Number of valid human captions:", len(captions_long))
print("Images missing quality information:", missing_quality)

display(
    master_metadata[
        [
            "file_name",
            "split",
            "text_detected",
            "BLR",
            "DRK",
            "NON",
            "unrecognizable_votes"
        ]
    ].head()
)

In [ ]:
master_metadata.to_csv(
    OUTPUT_FOLDER / "master_metadata.csv",
    index=False
)

captions_long.to_csv(
    OUTPUT_FOLDER / "captions_long.csv",
    index=False
)

print("Both files were saved.")

In [ ]:
# Make sure these columns contain numbers
for column in ["BLR", "DRK", "NON"]:
    master_metadata[column] = master_metadata[column].fillna(0)

master_metadata["text_detected"] = (
    master_metadata["text_detected"]
    .fillna(False)
    .astype(bool)
)

# These are all quality-problem columns in the dataset
quality_issue_columns = [
    "BLR",  # blurry
    "BRT",  # too bright
    "DRK",  # too dark
    "FRM",  # framing problem
    "OBS",  # obstruction
    "OTH",  # other problem
    "ROT"   # rotated image
]

master_metadata["highest_quality_issue_vote"] = (
    master_metadata[quality_issue_columns].max(axis=1)
)

# Start with no group assigned
master_metadata["candidate_group"] = ""

# Text candidates:
# Text was detected, and the image is not majority blurry or low-light
master_metadata.loc[
    (master_metadata["text_detected"]) &
    (master_metadata["BLR"] < 3) &
    (master_metadata["DRK"] < 3),
    "candidate_group"
] = "text"

# Blurry candidates:
# At least 3 of 5 marked blur; not low-light; no detected text
master_metadata.loc[
    (master_metadata["candidate_group"] == "") &
    (master_metadata["BLR"] >= 3) &
    (master_metadata["DRK"] < 3) &
    (~master_metadata["text_detected"]),
    "candidate_group"
] = "blurry"

# Low-light candidates:
# At least 3 of 5 marked darkness; not blurry; no detected text
master_metadata.loc[
    (master_metadata["candidate_group"] == "") &
    (master_metadata["DRK"] >= 3) &
    (master_metadata["BLR"] < 3) &
    (~master_metadata["text_detected"]),
    "candidate_group"
] = "low_light"

# Clear candidates:
# At least 3 of 5 marked no quality problem,
# and no quality problem received 3 or more votes
master_metadata.loc[
    (master_metadata["candidate_group"] == "") &
    (master_metadata["NON"] >= 3) &
    (master_metadata["highest_quality_issue_vote"] < 3) &
    (~master_metadata["text_detected"]),
    "candidate_group"
] = "clear"

# Show the number of available images in every group
print(master_metadata["candidate_group"].value_counts())

In [ ]:
SEED = 2026

quotas = {
    "clear": 70,
    "blurry": 70,
    "low_light": 60,
    "text": 50
}

selected_groups = []

for group_number, (group_name, number_needed) in enumerate(quotas.items()):

    # Get all eligible images for this group
    eligible_images = master_metadata[
        master_metadata["candidate_group"] == group_name
    ].copy()

    # Randomly choose the required number
    chosen_images = eligible_images.sample(
        n=number_needed,
        random_state=SEED + group_number
    ).copy()

    chosen_images["stratum"] = group_name
    selected_groups.append(chosen_images)

# Combine the four groups into one 250-image table
sample_manifest = pd.concat(selected_groups, ignore_index=True)

# Add columns your team will use during checking
sample_manifest["manual_text_review"] = ""
sample_manifest["expected_text"] = ""
sample_manifest["laplacian_variance"] = ""

# Keep the most useful columns
sample_manifest = sample_manifest[
    [
        "image_id",
        "file_name",
        "split",
        "stratum",
        "text_detected",
        "BLR",
        "DRK",
        "NON",
        "unrecognizable_votes",
        "manual_text_review",
        "expected_text",
        "laplacian_variance"
    ]
]

# Check that the sample is correct
print("Total selected images:", len(sample_manifest))
print("\nImages per group:")
print(sample_manifest["stratum"].value_counts())

print("\nDuplicate filenames:", sample_manifest["file_name"].duplicated().sum())

display(sample_manifest.head())

In [ ]:
sample_manifest.to_csv(
    OUTPUT_FOLDER / "sample_manifest_draft.csv",
    index=False
)

print("Draft sample saved.")

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

# Load your draft sample again
SAMPLE_FILE = OUTPUT_FOLDER / "sample_manifest_draft.csv"
sample_manifest = pd.read_csv(SAMPLE_FILE)

# Image folders
TRAIN_FOLDER = RAW_DATA / "train"
VAL_FOLDER = RAW_DATA / "val"

# Keep only the 50 text images
text_images = sample_manifest[
    sample_manifest["stratum"] == "text"
].reset_index(drop=True)

print("Number of text images to review:", len(text_images))


def show_text_images(start=0, count=6):
    """Shows a small page of text images for manual review."""

    page = text_images.iloc[start:start + count]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for axis, (_, row) in zip(axes, page.iterrows()):

        image_folder = TRAIN_FOLDER if row["split"] == "train" else VAL_FOLDER
        image_path = image_folder / row["file_name"]

        image = Image.open(image_path).convert("RGB")

        axis.imshow(image)
        axis.set_title(
            f"ID: {row['image_id']}\n{row['file_name']}",
            fontsize=9
        )
        axis.axis("off")

    # Hide unused spaces
    for axis in axes[len(page):]:
        axis.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
def show_five_text_images(start=0):
    """Shows five text images at a time for clear manual review."""

    page = text_images.iloc[start:start + 5]

    fig, axes = plt.subplots(1, len(page), figsize=(24, 7))

    # Makes the function work even if there is only one image
    if len(page) == 1:
        axes = [axes]

    for axis, (_, row) in zip(axes, page.iterrows()):

        image_folder = TRAIN_FOLDER if row["split"] == "train" else VAL_FOLDER
        image_path = image_folder / row["file_name"]

        image = Image.open(image_path).convert("RGB")

        axis.imshow(image)
        axis.set_title(
            f"ID: {row['image_id']}\n{row['file_name']}",
            fontsize=10
        )
        axis.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_five_text_images(0)    # Images 1–5

In [ ]:
show_five_text_images(5)    # Images 6–10

In [ ]:
show_five_text_images(10)   # Images 11–15

In [ ]:
show_five_text_images(15)   # Images 16–20

In [ ]:
show_five_text_images(20)   # Images 21–25

In [ ]:
show_five_text_images(25)   # Images 26–30

In [ ]:
show_five_text_images(30)   # Images 31–35

In [ ]:
show_five_text_images(35)   # Images 36–40

In [ ]:
show_five_text_images(40)   # Images 41–45

In [ ]:
show_five_text_images(45)   # Images 46–50

In [ ]:
from google.colab import files
import io
import pandas as pd

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload only the reviewed Excel workbook.")

uploaded_name = next(iter(uploaded))

reviewed = pd.read_excel(
    io.BytesIO(uploaded[uploaded_name]),
    sheet_name="sample_manifest_draft"
)

# Standardize capitalization and remove extra spaces for counting.
# Your original review entries remain unchanged.
decisions = (
    reviewed["manual_text_review"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

is_text = reviewed["stratum"].eq("text")

assert reviewed["file_name"].is_unique, "Duplicate image filenames found."
assert is_text.sum() == 50, "Expected the original 50 text images."
assert decisions[is_text].isin(["approved", "rejected"]).all(), (
    "Some text images need an approved or rejected decision."
)

approved_text = reviewed[is_text & decisions.eq("approved")].copy()
rejected_text = reviewed[is_text & decisions.eq("rejected")].copy()

print("Approved:", len(approved_text))
print("Rejected:", len(rejected_text))

In [ ]:
from pathlib import Path

PROJECT_FOLDER = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project"
)

OUTPUT_FOLDER = PROJECT_FOLDER / "outputs"
MASTER_FILE = OUTPUT_FOLDER / "master_metadata.csv"
ROUND_FOLDER = OUTPUT_FOLDER / "text_review_round_2"

master = pd.read_csv(MASTER_FILE)

assert master["file_name"].is_unique, (
    "The master table contains duplicate filenames."
)

# Read True/False safely, whether stored as words or Boolean values.
text_flag = (
    master["text_detected"]
    .astype(str)
    .str.strip()
    .str.lower()
)

assert text_flag.isin(["true", "false"]).all(), (
    "Some text-detection values are missing or unexpected."
)

blur_votes = pd.to_numeric(master["BLR"], errors="raise")
dark_votes = pd.to_numeric(master["DRK"], errors="raise")

assert blur_votes.notna().all() and dark_votes.notna().all(), (
    "Some quality votes are missing."
)

# Apply the same eligibility rule used for the original text group.
eligible = master[
    master["split"].isin(["train", "val"])
    & text_flag.eq("true")
    & blur_votes.lt(3)
    & dark_votes.lt(3)
    & ~master["file_name"].isin(reviewed["file_name"])
].copy()

number_needed = 50 - len(approved_text)

assert len(eligible) >= number_needed, "Not enough eligible replacements."

# Sorting and a fixed seed make this selection repeatable.
replacements = (
    eligible.sort_values("file_name")
    .sample(n=number_needed, random_state=2027)
    .copy()
)

replacements["stratum"] = "text"
replacements["manual_text_review"] = ""
replacements["expected_text"] = ""
replacements["laplacian_variance"] = ""

# Match the columns in your reviewed workbook.
replacements = replacements.reindex(columns=reviewed.columns)

# Preserve earlier review files if this cell is run again.
if ROUND_FOLDER.exists():
    raise FileExistsError(
        "text_review_round_2 already exists. "
        "Use its saved replacement_candidates.csv; do not overwrite your reviews."
    )

ROUND_FOLDER.mkdir(parents=True)

# Save the complete first-round record, including rejected images.
reviewed.to_csv(
    ROUND_FOLDER / "original_review_record.csv", index=False
)

# Save the 200 other images plus the 29 approved text images.
retained = reviewed[~is_text | decisions.eq("approved")].copy()
retained.to_csv(
    ROUND_FOLDER / "retained_sample.csv", index=False
)

replacements.to_csv(
    ROUND_FOLDER / "replacement_candidates.csv", index=False
)

print("Images retained:", len(retained))
print("New text candidates:", len(replacements))
print("Saved in:", ROUND_FOLDER)

In [ ]:
from PIL import Image, ImageOps
from IPython.display import display

replacements = pd.read_csv(
    ROUND_FOLDER / "replacement_candidates.csv"
)

def show_replacements(start=0):
    page = replacements.iloc[start:start + 5]

    for position, row in page.iterrows():
        image_path = (
            PROJECT_FOLDER
            / "raw_data"
            / row["split"]
            / row["file_name"]
        )

        print(
            f"Replacement {position + 1} of {len(replacements)}"
            f" | ID: {row['image_id']}"
        )
        print(row["file_name"])

        with Image.open(image_path) as original:
            picture = ImageOps.exif_transpose(original).convert("RGB")
            picture.thumbnail((1000, 1000))
            display(picture)

        print()

In [ ]:
show_replacements(0)    # Replacements 1–5

In [ ]:
show_replacements(5)    # Replacements 6–10

In [ ]:
show_replacements(10)   # Replacements 11–15

In [ ]:
show_replacements(15)   # Replacements 16–20

In [ ]:
show_replacements(20)   # Replacement 21

In [ ]:
from google.colab import files
from pathlib import Path
import io
import pandas as pd

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Upload only replacement_candidates.xlsx.")

uploaded_name = next(iter(uploaded))

round2_review = pd.read_excel(
    io.BytesIO(uploaded[uploaded_name]),
    sheet_name="replacement_candidates"
)

print("Replacement images loaded:", len(round2_review))

In [ ]:
PROJECT_FOLDER = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project"
)
OUTPUT_FOLDER = PROJECT_FOLDER / "outputs"
ROUND3_FOLDER = OUTPUT_FOLDER / "text_review_round_3"

# Load the original 250-image review record.
original_review = pd.read_csv(
    OUTPUT_FOLDER / "text_review_round_2" / "original_review_record.csv"
)

# Combine all images selected or reviewed so far.
history = pd.concat(
    [original_review, round2_review],
    ignore_index=True
)

assert len(original_review) == 250
assert len(round2_review) == 21
assert history["file_name"].is_unique, "Duplicate image filenames found."

# Preserve your entries and put standardized decisions in a new column.
history["review_status"] = (
    history["manual_text_review"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "accepted": "approved",
        "rejecetd": "rejected"
    })
)

text_rows = history["stratum"].eq("text")

assert history.loc[text_rows, "review_status"].isin(
    ["approved", "rejected"]
).all(), "Some text-review decisions are missing or unrecognized."

approved_text = history[
    text_rows & history["review_status"].eq("approved")
].copy()

# Keep the other 200 images and every approved text image.
retained = history[
    ~text_rows | history["review_status"].eq("approved")
].copy()

number_needed = 50 - len(approved_text)
assert number_needed > 0, "You already have 50 approved text images."

# Load the full image list.
master = pd.read_csv(OUTPUT_FOLDER / "master_metadata.csv")
assert master["file_name"].is_unique

text_flag = (
    master["text_detected"].astype(str).str.strip().str.lower()
)
assert text_flag.isin(["true", "false"]).all()

blur = pd.to_numeric(master["BLR"], errors="raise")
dark = pd.to_numeric(master["DRK"], errors="raise")
assert blur.notna().all() and dark.notna().all()

# Same text-candidate rules as before.
# Exclude ALL previously selected images, approved or rejected.
eligible = master[
    master["split"].isin(["train", "val"])
    & text_flag.eq("true")
    & blur.lt(3)
    & dark.lt(3)
    & ~master["file_name"].isin(history["file_name"])
].copy()

assert len(eligible) >= number_needed

replacements = (
    eligible.sort_values("file_name")
    .sample(n=number_needed, random_state=2028)
    .copy()
)

replacements["stratum"] = "text"
replacements["manual_text_review"] = ""
replacements["expected_text"] = ""
replacements["laplacian_variance"] = ""
replacements = replacements.reindex(
    columns=original_review.columns
).reset_index(drop=True)

# Prevent accidentally overwriting a review you have started.
if ROUND3_FOLDER.exists():
    raise FileExistsError(
        "Round 3 already exists. Use its saved replacement_candidates.csv."
    )

ROUND3_FOLDER.mkdir(parents=True)

history.to_csv(
    ROUND3_FOLDER / "review_history.csv", index=False
)
retained.to_csv(
    ROUND3_FOLDER / "retained_sample.csv", index=False
)
replacements.to_csv(
    ROUND3_FOLDER / "replacement_candidates.csv", index=False
)

print("Approved text images so far:", len(approved_text))
print("Total images retained:", len(retained))
print("New candidates to review:", len(replacements))
print("Saved in:", ROUND3_FOLDER)

In [ ]:
show_replacements(0)    # New candidates 1–5

In [ ]:
show_replacements(5)    # New candidates 6–10

In [ ]:
show_replacements(10)   # New candidates 11–13

In [ ]:
from google.colab import files
from pathlib import Path
import io
import pandas as pd

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload only the latest reviewed workbook.")

uploaded_name = next(iter(uploaded))

latest_review = pd.read_excel(
    io.BytesIO(uploaded[uploaded_name]),
    sheet_name="replacement_candidates"
)

print("Images loaded:", len(latest_review))

In [ ]:
PROJECT_FOLDER = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project"
)
OUTPUT_FOLDER = PROJECT_FOLDER / "outputs"
NEW_FOLDER = OUTPUT_FOLDER / "text_review_round_4"

# Load the history saved before your latest 13-image review.
previous_history = pd.read_csv(
    OUTPUT_FOLDER / "text_review_round_3" / "review_history.csv"
)

# Add the latest reviewed images.
history = pd.concat(
    [previous_history, latest_review],
    ignore_index=True
)

assert len(latest_review) == 13, "Expected the latest 13-image batch."
assert history["file_name"].is_unique, (
    "An image appears more than once in the combined history."
)

# Standardize decisions without changing your original entries.
history["review_status"] = (
    history["manual_text_review"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "accepted": "approved",
        "rejecetd": "rejected"
    })
)

is_text = history["stratum"].eq("text")

assert history.loc[is_text, "review_status"].isin(
    ["approved", "rejected"]
).all(), "Some review decisions are missing or unrecognized."

approved_count = (
    is_text & history["review_status"].eq("approved")
).sum()

# Retain the 200 non-text images and all approved text images.
retained = history[
    ~is_text | history["review_status"].eq("approved")
].copy()

# Load the full master table.
master = pd.read_csv(OUTPUT_FOLDER / "master_metadata.csv")
assert master["file_name"].is_unique

text_flag = (
    master["text_detected"].astype(str).str.strip().str.lower()
)
assert text_flag.isin(["true", "false"]).all()

blur = pd.to_numeric(master["BLR"], errors="raise")
dark = pd.to_numeric(master["DRK"], errors="raise")
assert blur.notna().all() and dark.notna().all()

# Apply the same text-candidate rules.
# Exclude every image selected in any previous round.
eligible = master[
    master["split"].isin(["train", "val"])
    & text_flag.eq("true")
    & blur.lt(3)
    & dark.lt(3)
    & ~master["file_name"].isin(history["file_name"])
].copy()

BATCH_SIZE = 30
assert len(eligible) >= BATCH_SIZE, "Not enough unused candidates."

replacements = (
    eligible.sort_values("file_name")
    .sample(n=BATCH_SIZE, random_state=2029)
    .copy()
)

replacements["stratum"] = "text"
replacements["manual_text_review"] = ""
replacements["expected_text"] = ""
replacements["laplacian_variance"] = ""

replacements = replacements.reindex(
    columns=latest_review.columns
).reset_index(drop=True)

# Protect any review work already saved in this folder.
if NEW_FOLDER.exists():
    raise FileExistsError(
        "Round 4 already exists. Use its saved replacement_candidates.csv."
    )

NEW_FOLDER.mkdir(parents=True)

history.to_csv(
    NEW_FOLDER / "review_history.csv", index=False
)
retained.to_csv(
    NEW_FOLDER / "retained_sample.csv", index=False
)
replacements.to_csv(
    NEW_FOLDER / "replacement_candidates.csv", index=False
)

print("Approved text images so far:", approved_count)
print("Additional approved images needed:", 50 - approved_count)
print("New candidates available for review:", len(replacements))
print("Saved in:", NEW_FOLDER)

In [ ]:
show_replacements(0)    # Candidates 1–5

In [ ]:
show_replacements(5)    # Candidates 6–10

In [ ]:
show_replacements(10)   # Candidates 11–15

In [ ]:
show_replacements(15)   # Candidates 16–20

In [ ]:
show_replacements(20)   # Candidates 21–25

In [ ]:
show_replacements(25)   # Candidates 26–30

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image

PROJECT_FOLDER = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project"
)

SAMPLE_FILE = (
    PROJECT_FOLDER / "outputs/final_sample/sample_manifest.csv"
)

if not SAMPLE_FILE.exists():
    raise FileNotFoundError(
        "Upload the consolidated sample_manifest.csv into "
        "outputs/final_sample first."
    )

sample = pd.read_csv(SAMPLE_FILE)

# Check the number of images and the group sizes.
expected_counts = {
    "clear": 70,
    "blurry": 70,
    "low_light": 60,
    "text": 50
}

assert len(sample) == 250, "The sample should contain 250 rows."
assert sample["file_name"].is_unique, "Duplicate filenames found."
assert sample["split"].isin(["train", "val"]).all()
assert sample["stratum"].value_counts().to_dict() == expected_counts, (
    "The image-group counts do not match the agreed sample."
)

problems = []
opened = 0

for _, row in sample.iterrows():
    image_path = (
        PROJECT_FOLDER / "raw_data"
        / row["split"] / row["file_name"]
    )

    if not image_path.is_file():
        problems.append({
            "file_name": row["file_name"],
            "problem": "File not found at the expected location"
        })
        continue

    try:
        with Image.open(image_path) as image:
            image.load()  # Read the image pixels.
        opened += 1
    except Exception as error:
        problems.append({
            "file_name": row["file_name"],
            "problem": str(error)
        })

print("Images listed:", len(sample))
print("Images successfully opened:", opened)
print("Images needing attention:", len(problems))
print("\nGroup counts:")
print(sample["stratum"].value_counts())

if problems:
    display(pd.DataFrame(problems))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
from PIL import Image, ImageOps

PROJECT_FOLDER = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project"
)
FINAL_FOLDER = PROJECT_FOLDER / "outputs/final_sample"

sample = pd.read_csv(FINAL_FOLDER / "sample_manifest.csv")

# Our initial processing choice:
# resize a working copy so its longest side is 512 pixels.
# Original image files are not changed.
LONGEST_SIDE = 512

scores = []

for number, (_, row) in enumerate(sample.iterrows(), start=1):

    image_path = (
        PROJECT_FOLDER / "raw_data"
        / row["split"] / row["file_name"]
    )

    with Image.open(image_path) as original:
        picture = ImageOps.exif_transpose(original).convert("RGB")
        pixels = np.array(picture)

    gray = cv2.cvtColor(pixels, cv2.COLOR_RGB2GRAY)

    height, width = gray.shape
    scale = LONGEST_SIDE / max(height, width)

    new_width = max(1, round(width * scale))
    new_height = max(1, round(height * scale))

    interpolation = (
        cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    )

    resized = cv2.resize(
        gray,
        (new_width, new_height),
        interpolation=interpolation
    )

    laplacian = cv2.Laplacian(
        resized,
        cv2.CV_64F,
        ksize=1,
        borderType=cv2.BORDER_REFLECT_101
    )

    scores.append(float(laplacian.var()))

    if number % 50 == 0:
        print(f"Processed {number} of {len(sample)} images")

sample["laplacian_variance"] = scores

assert len(sample) == 250
assert np.isfinite(sample["laplacian_variance"]).all()

print("\nScores calculated:", len(scores))

In [ ]:
import json

# Save a new file, preserving the original sample manifest.
sample.to_csv(
    FINAL_FOLDER / "sample_manifest_with_blur.csv",
    index=False
)

# Record the settings so the calculation can be repeated.
settings = {
    "method": "variance of grayscale Laplacian",
    "orientation": "PIL ImageOps.exif_transpose",
    "grayscale": "OpenCV COLOR_RGB2GRAY",
    "longest_side_pixels": 512,
    "aspect_ratio_preserved": True,
    "resize_down": "INTER_AREA",
    "resize_up": "INTER_LINEAR",
    "laplacian_depth": "CV_64F",
    "laplacian_kernel_size": 1,
    "border": "BORDER_REFLECT_101",
    "variance_ddof": 0,
    "additional_smoothing": False,
    "opencv_version": cv2.__version__,
    "numpy_version": np.__version__,
    "threshold": None
}

with open(FINAL_FOLDER / "blur_settings.json", "w") as file:
    json.dump(settings, file, indent=2)

summary = (
    sample.groupby("stratum")["laplacian_variance"]
    .agg(
        images="count",
        median_score="median",
        average_score="mean",
        lowest_score="min",
        highest_score="max"
    )
)

display(summary.round(2))
print("Saved sample_manifest_with_blur.csv and blur_settings.json")

In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display

PROJECT_FOLDER = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project"
)

scored_sample = pd.read_csv(
    PROJECT_FOLDER
    / "outputs/final_sample/sample_manifest_with_blur.csv"
)

def inspect_blur_scores(group, lowest=True, count=3):
    examples = (
        scored_sample[scored_sample["stratum"].eq(group)]
        .sort_values(
            ["laplacian_variance", "file_name"],
            ascending=[lowest, True]
        )
        .head(count)
    )

    for _, row in examples.iterrows():
        print(f"Group: {row['stratum']}")
        print(f"Image ID: {row['image_id']}")
        print(f"Filename: {row['file_name']}")
        print(f"Blur score: {row['laplacian_variance']:.2f}")
        print(
            f"Existing votes — Blur: {row['BLR']}, "
            f"Darkness: {row['DRK']}, "
            f"No quality problem: {row['NON']}"
        )

        image_path = (
            PROJECT_FOLDER / "raw_data"
            / row["split"] / row["file_name"]
        )

        with Image.open(image_path) as original:
            picture = ImageOps.exif_transpose(original).convert("RGB")
            picture.thumbnail((1000, 1000))
            display(picture)

        print()

In [ ]:
inspect_blur_scores("clear", lowest=True)

In [ ]:
inspect_blur_scores("blurry", lowest=False)

In [ ]:
inspect_blur_scores("low_light", lowest=True)

In [ ]:
from pathlib import Path
import pandas as pd

FINAL_FOLDER = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project"
    "/outputs/final_sample"
)

observations = [
    (3221, "blurry", 567.62,
     "The image looks blurry; a Christmas tree.", "yes"),

    (11636, "blurry", 566.98,
     "The image looks blurry; a plant/tree.", "yes"),

    (7125, "blurry", 388.18,
     "The image looks blurry; a shirt.", "yes"),

    (24897, "low_light", 0.24,
     "The image has low light; it is dark.", "yes"),

    (22558, "low_light", 1.86,
     "The image has low light; it is dark.", "yes"),

    (13757, "low_light", 2.48,
     "The image has low light; it is dark.", "yes"),

    (16651, "clear", 16.89,
     "The image looks sharp; fruit slices of cantaloupe.", "yes"),

    (19572, "clear", 29.48,
     "The image looks sharp; an iPhone.", "yes"),

    (16174, "clear", 34.93,
     "The image does not look sharp; it is zoomed in and blurry.", "no")
]

review = pd.DataFrame(
    observations,
    columns=[
        "image_id",
        "existing_group",
        "displayed_blur_score",
        "observation",
        "existing_label_seems_reasonable"
    ]
)

review["reviewers"] = "Ebere and Samuel"
review["selection_method"] = "Selected score extremes; not random"

# Use a separate file so the sample labels remain unchanged.
REVIEW_FILE = FINAL_FOLDER / "blur_inspection_notes.csv"

if REVIEW_FILE.exists():
    print("The notes file already exists. Check it before replacing it.")
else:
    review.to_csv(REVIEW_FILE, index=False)
    print("Saved:", REVIEW_FILE)

In [ ]:
scored_sample = pd.read_csv(
    FINAL_FOLDER / "sample_manifest_with_blur.csv"
)

display(
    scored_sample.loc[
        scored_sample["image_id"].eq(16174),
        [
            "image_id",
            "file_name",
            "stratum",
            "BLR",
            "DRK",
            "NON",
            "unrecognizable_votes",
            "laplacian_variance"
        ]
    ]
)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_FOLDER = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project"
)
OUTPUT_FOLDER = PROJECT_FOLDER / "outputs"
PILOT_FOLDER = OUTPUT_FOLDER / "blur_calibration"
PILOT_FILE = PILOT_FOLDER / "pilot_manifest.csv"

master = pd.read_csv(OUTPUT_FOLDER / "master_metadata.csv")
main_sample = pd.read_csv(
    OUTPUT_FOLDER / "final_sample/sample_manifest.csv"
)

assert master["file_name"].is_unique

# Exclude the main 250 images.
excluded = set(main_sample["file_name"])

# Also exclude earlier reviewed candidates if the combined history
# has been uploaded into final_sample.
history_file = (
    OUTPUT_FOLDER / "final_sample/review_history_combined.csv"
)

if history_file.exists():
    history = pd.read_csv(history_file)
    excluded.update(history["file_name"])

# Read text flags and quality votes without replacing missing data.
text_flag = (
    master["text_detected"].astype(str).str.strip().str.lower()
)
assert text_flag.isin(["true", "false"]).all()

issue_columns = ["BLR", "BRT", "DRK", "FRM", "OBS", "OTH", "ROT"]

for column in issue_columns + ["NON"]:
    master[column] = pd.to_numeric(master[column], errors="raise")
    assert master[column].notna().all(), f"Missing votes in {column}"

available = (
    master["split"].isin(["train", "val"])
    & ~master["file_name"].isin(excluded)
    & text_flag.eq("false")
)

# Use the same clear and blurry rules as the main sample.
clear_pool = master[
    available
    & master["NON"].ge(3)
    & master[issue_columns].max(axis=1).lt(3)
].copy()

blurry_pool = master[
    available
    & master["BLR"].ge(3)
    & master["DRK"].lt(3)
].copy()

assert len(clear_pool) >= 100
assert len(blurry_pool) >= 100

# Reuse an existing pilot instead of silently drawing a new one.
if PILOT_FILE.exists():
    pilot = pd.read_csv(PILOT_FILE)
    print("Loaded the previously saved pilot.")
else:
    parts = []

    for group, pool, seed in [
        ("clear", clear_pool, 2031),
        ("blurry", blurry_pool, 2032)
    ]:
        chosen = (
            pool.sort_values("file_name")
            .sample(n=100, random_state=seed)
            .reset_index(drop=True)
        )

        chosen["reference_group"] = group
        chosen["purpose"] = (
            ["choose_threshold"] * 50
            + ["check_threshold"] * 50
        )
        chosen["sampling_seed"] = seed
        parts.append(chosen)

    pilot = pd.concat(parts, ignore_index=True)

    PILOT_FOLDER.mkdir(parents=True, exist_ok=True)
    pilot.to_csv(PILOT_FILE, index=False)
    print("Saved the new pilot.")

assert len(pilot) == 200
assert pilot["file_name"].is_unique
assert not pilot["file_name"].isin(excluded).any()

print("\nPilot images:", len(pilot))
display(
    pd.crosstab(pilot["purpose"], pilot["reference_group"])
)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
from PIL import Image, ImageOps

PROJECT_FOLDER = Path(
    "/content/drive/MyDrive/NAIJA JOLLOF/BLV Project"
)
PILOT_FOLDER = PROJECT_FOLDER / "outputs/blur_calibration"

pilot = pd.read_csv(PILOT_FOLDER / "pilot_manifest.csv")

assert len(pilot) == 200
assert pilot["file_name"].is_unique

scores = []

for number, (_, row) in enumerate(pilot.iterrows(), start=1):
    image_path = (
        PROJECT_FOLDER / "raw_data"
        / row["split"] / row["file_name"]
    )

    with Image.open(image_path) as original:
        picture = ImageOps.exif_transpose(original).convert("RGB")
        pixels = np.array(picture)

    gray = cv2.cvtColor(pixels, cv2.COLOR_RGB2GRAY)

    height, width = gray.shape
    scale = 512 / max(height, width)

    resized = cv2.resize(
        gray,
        (
            max(1, round(width * scale)),
            max(1, round(height * scale))
        ),
        interpolation=(
            cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
        )
    )

    laplacian = cv2.Laplacian(
        resized,
        cv2.CV_64F,
        ksize=1,
        borderType=cv2.BORDER_REFLECT_101
    )

    scores.append(float(laplacian.var()))

    if number % 50 == 0:
        print(f"Processed {number} of {len(pilot)} images")

pilot["laplacian_variance"] = scores

assert np.isfinite(pilot["laplacian_variance"]).all()

pilot.to_csv(
    PILOT_FOLDER / "pilot_with_blur_scores.csv",
    index=False
)

print("Saved blur scores for all 200 pilot images.")

In [ ]:
import json

calibration = pilot[
    pilot["purpose"].eq("choose_threshold")
].copy()

assert calibration["reference_group"].value_counts().to_dict() == {
    "clear": 50,
    "blurry": 50
}

values = calibration["laplacian_variance"].to_numpy()
actual_blurry = calibration["reference_group"].eq("blurry").to_numpy()

# Test cutoffs between distinct scores, plus the two extremes.
unique_scores = np.sort(np.unique(values))

midpoints = (
    unique_scores[:-1]
    + (unique_scores[1:] - unique_scores[:-1]) / 2
)

thresholds = np.concatenate([
    [0.0],
    midpoints,
    [np.nextafter(unique_scores[-1], np.inf)]
])

results = []

for threshold in thresholds:
    predicted_blurry = values < threshold

    blurry_recall = predicted_blurry[actual_blurry].mean()
    clear_recall = (~predicted_blurry[~actual_blurry]).mean()

    results.append({
        "threshold": float(threshold),
        "balanced_accuracy": float(
            (blurry_recall + clear_recall) / 2
        )
    })

search_results = pd.DataFrame(results)
best_accuracy = search_results["balanced_accuracy"].max()

best = search_results[
    np.isclose(search_results["balanced_accuracy"], best_accuracy)
].sort_values("threshold")

# Fixed tie rule: choose the middle candidate among tied best cutoffs.
chosen = best.iloc[len(best) // 2]
THRESHOLD = float(chosen["threshold"])

settings = {
    "threshold": THRESHOLD,
    "prediction_rule": "blurry if score < threshold; otherwise clear",
    "selection_metric": "balanced_accuracy",
    "tie_rule": "middle candidate among tied best thresholds",
    "calibration_images": len(calibration),
    "calibration_balanced_accuracy": float(best_accuracy),
    "longest_side_pixels": 512,
    "opencv_version": cv2.__version__,
    "numpy_version": np.__version__
}

with open(PILOT_FOLDER / "selected_threshold.json", "w") as file:
    json.dump(settings, file, indent=2)

search_results.to_csv(
    PILOT_FOLDER / "threshold_search.csv",
    index=False
)

print(f"Selected threshold: {THRESHOLD:.6f}")
print(f"Calibration balanced accuracy: {best_accuracy:.1%}")

In [ ]:
with open(PILOT_FOLDER / "selected_threshold.json") as file:
    frozen_settings = json.load(file)

threshold = frozen_settings["threshold"]

check = pilot[
    pilot["purpose"].eq("check_threshold")
].copy()

assert check["reference_group"].value_counts().to_dict() == {
    "clear": 50,
    "blurry": 50
}

check["predicted_group"] = np.where(
    check["laplacian_variance"] < threshold,
    "blurry",
    "clear"
)

check["matches_reference"] = (
    check["predicted_group"] == check["reference_group"]
)

comparison = pd.crosstab(
    check["reference_group"],
    check["predicted_group"]
).reindex(
    index=["blurry", "clear"],
    columns=["blurry", "clear"],
    fill_value=0
)

group_accuracy = check.groupby(
    "reference_group"
)["matches_reference"].mean()

balanced_accuracy = group_accuracy.mean()

check.to_csv(
    PILOT_FOLDER / "threshold_check_results.csv",
    index=False
)

print("Rows = existing labels; columns = predicted labels")
display(comparison)

print(
    f"Blurry images correctly identified: "
    f"{group_accuracy['blurry']:.1%}"
)
print(
    f"Clear images correctly identified: "
    f"{group_accuracy['clear']:.1%}"
)
print(f"Check balanced accuracy: {balanced_accuracy:.1%}")